In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import time
import os 
import sys
sys.path.append('..')


In [7]:
data_train = pd.read_parquet('../temp/X_train_v1.parquet')
data_val = pd.read_parquet('../temp/X_val_v1.parquet')



In [8]:
data_train.shape

(452266, 895)

In [9]:
data_val.shape

(61502, 895)

In [10]:
X_train = data_train.iloc[1:100001]
X_val = data_val.iloc[1:10001]

In [12]:
y_train = X_train['TARGET']
y_val = X_val['TARGET']
X_train = X_train.drop(columns=['TARGET'])
X_val = X_val.drop(columns=['TARGET'])

In [ ]:


sample_fraction = 0.2  
random_state = 42
n_features_to_select = 500
step_size = 50 

total_features = X_train.shape[1]
print(f"Total number of features: {total_features}")

if total_features <= 1000:
    step_size = 10
elif total_features <= 5000:
    step_size = 25
else:
    step_size = 50

print(f"Using step_size: {step_size}")

X_sample, _, y_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=sample_fraction,
    stratify=y_train,
    random_state=random_state
)

print(f"Original training set size: {X_train.shape[0]} samples")
print(f"Sampled data size for RFE: {X_sample.shape[0]} samples")

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rfe', RFE(
        estimator=LogisticRegression(
            penalty='l2',
            solver='saga',
            max_iter=1000,
            n_jobs=-1,
            random_state=random_state,
            warm_start=True 
        ),
        n_features_to_select=n_features_to_select,
        step=step_size,
        verbose=1
    ))
])

print("Starting RFE...")
start_time = time.time()
pipeline.fit(X_sample, y_sample)
end_time = time.time()
print(f"RFE completed in {end_time - start_time:.2f} seconds.")

selected_features_mask = pipeline.named_steps['rfe'].support_
print(f"Number of selected features: {selected_features_mask.sum()}")

X_train_selected = pipeline.transform(X_train)
X_test_selected = pipeline.transform(X_test)

final_model = LogisticRegression(
    penalty='l2',
    solver='saga',
    max_iter=1000,
    n_jobs=-1,
    random_state=random_state
)

final_model.fit(X_train_selected, y_train)

y_pred = final_model.predict(X_test_selected)
accuracy = accuracy_score(y_test, y_pred)
print(f"Final model accuracy with selected features: {accuracy:.4f}")

if isinstance(X_train, pd.DataFrame):
    selected_feature_names = X_train.columns[selected_features_mask]
    print("Selected Features:")
    print(selected_feature_names.tolist())
    selected_feature_names.to_csv('selected_features.csv', index=False)
else:
    feature_indices = np.where(selected_features_mask)[0]
    print("Selected Feature Indices:")
    print(feature_indices.tolist())
    np.savetxt('selected_feature_indices.txt', feature_indices, fmt='%d')
